# Neighbour demand and spatial shocks

Construct boundary-contact adjacency, previous-hour neighbour demand, and past-only neighbour shocks. Both point and segment contacts count as neighbours.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import pandas as pd
import numpy as np
import geopandas as gpd
import data_tools as data
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer


In [ ]:
import importlib
importlib.reload(data)


In [ ]:
df_zones = gpd.read_file(str(project_root / "taxi_zones/taxi_zones.shp"))
df_general = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_features.parquet"))

In [ ]:
X = df_general.copy()


In [ ]:
df_zones.plot()


In [ ]:
boundary = df_zones.geometry.iloc[6].boundary
shared_parts = df_zones.boundary.intersection(boundary)

mask = ~shared_parts.is_empty
neighbors = df_zones.loc[mask].drop(index = df_zones.index[6])


In [ ]:
neighbors


In [ ]:
adjacency = np.zeros((263, 263))
for i in range(263):
    boundary = df_zones.geometry.iloc[i].boundary
    shared_parts = df_zones.boundary.intersection(boundary)
    mask = ~shared_parts.is_empty
    neighbors = df_zones.loc[mask].drop(index = df_zones.index[i])
    for j in neighbors.LocationID:
        adjacency[i][j-1] += 1

adjacency


In [ ]:
pickups = X.pivot(
    index = "tpep_pickup_datetime",
    columns = "PULocationID",
    values = "trip_count"
).sort_index().reindex(columns = range(1, 264))

# Previous-hour demand; your dataset has consecutive hourly rows.
previous = pickups.shift(1)

neighbor_counts = adjacency.sum(axis = 1)
neighbor_means = previous.to_numpy() @ adjacency.T

# Average over neighbours; keep your zero fallback for isolated zones.
has_neighbors = neighbor_counts > 0
neighbor_means[:, has_neighbors] /= neighbor_counts[has_neighbors]
neighbor_means[:, ~has_neighbors] = 0

neighbor_table = pd.DataFrame(
    neighbor_means,
    index = pickups.index,
    columns = pickups.columns
)

row_positions = neighbor_table.index.get_indexer(X["tpep_pickup_datetime"])
col_positions = neighbor_table.columns.get_indexer(X["PULocationID"])

X["neighbors_mean_PU"] = neighbor_means[row_positions, col_positions]


In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, train, val, test = data.split(X)


In [ ]:
train_ablation = train.drop(columns = ["month"])
#xgboost
categorical = ["PULocationID", "hours"]
numerical = [
    col for col in train_ablation.columns
    if col not in categorical
]


preprocessor = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown = "ignore"), categorical),
    ("numbers", StandardScaler(), numerical),
])
model_spatial = make_pipeline(
    preprocessor,
    xgb.XGBRegressor(
        objective = "reg:squarederror",
        random_state = 42,

    )
)

model_spatial.fit(train_ablation, y_train)


In [ ]:
y_pred = model_spatial.predict(val)

data.metrics(y_val, y_pred)


In [ ]:
val_high, val_mid, val_low, y_high, y_pred_high, y_mid, y__pred_mid, y_low, y_pred_low = data.ranking_zones(X_train, X_val, val, model_spatial)


In [ ]:
#high
data.metrics(y_high, y_pred_high)


In [ ]:
#mid
data.metrics(y_mid, y__pred_mid)


In [ ]:
#low
data.metrics(y_low, y_pred_low)


In [ ]:
from sklearn.base import clone

# Keep the same preprocessing and model settings; remove only spatial mean.
train_without_spatial = train_ablation.drop(columns = ["neighbors_mean_PU"])
model_without_spatial = clone(model_spatial)
preprocessor_without_spatial = model_without_spatial.named_steps["columntransformer"]
preprocessor_without_spatial.set_params(transformers = [
    (name, transformer, [col for col in columns if col != "neighbors_mean_PU"])
    for name, transformer, columns in preprocessor_without_spatial.transformers
])
model_without_spatial.fit(train_without_spatial, y_train)

assert val.index.equals(y_val.index)
assert X_val.index.equals(val.index)
weekly_predictions = pd.DataFrame({
    "timestamp": X_val["tpep_pickup_datetime"],
    "actual": y_val,
    "with_spatial": model_spatial.predict(val),
    "without_spatial": model_without_spatial.predict(val[train_without_spatial.columns]),
}, index = val.index)

# September 1–7, 8–14, 15–21, 22–28, and the final partial week 29–30.
september_start = weekly_predictions["timestamp"].min().to_period("M").start_time
september_end = september_start + pd.DateOffset(months = 1)
weekly_results = []
for week_start in pd.date_range(september_start, september_end, freq = "7D", inclusive = "left"):
    week_end = min(week_start + pd.Timedelta(days = 7), september_end)
    rows = weekly_predictions.loc[
        (weekly_predictions["timestamp"] >= week_start)
        & (weekly_predictions["timestamp"] < week_end)
    ]
    weekly_results.append({
        "September dates": f"{week_start:%d}–{week_end - pd.Timedelta(days = 1):%d}",
        "Hours": rows["timestamp"].nunique(),
        "MAE with spatial": mean_absolute_error(rows["actual"], rows["with_spatial"]),
        "MAE without spatial": mean_absolute_error(rows["actual"], rows["without_spatial"]),
        "RMSE with spatial": root_mean_squared_error(rows["actual"], rows["with_spatial"]),
        "RMSE without spatial": root_mean_squared_error(rows["actual"], rows["without_spatial"]),
    })

weekly_metrics = pd.DataFrame(weekly_results).set_index("September dates")
print("September validation — identical model settings; month excluded from both models")
print("Both models are trained on January–August. Lower MAE and RMSE are better.")
print(weekly_metrics.to_string(float_format = lambda value: f"{value:.4f}"))


#### Spatial shock


In [ ]:
# Average neighbouring demand deviations at exactly t - 1 hour.
# Baseline: empirical mean for each zone and hour of day.
shock_training_end = pd.Timestamp("2024-09-01")
shock_pickups = X.pivot(
    index = "tpep_pickup_datetime", columns = "PULocationID", values = "trip_count"
).sort_index().reindex(columns = range(1, 264))

# Training baselines use only earlier observations of the same hour of day.
# This excludes the current target and future training demand.
past_hour_means = shock_pickups.groupby(shock_pickups.index.hour).transform(
    lambda counts: counts.expanding().mean().shift(1)
)

# Freeze the evaluation baseline using January–August only.
training_pickups = shock_pickups.loc[shock_pickups.index < shock_training_end]
zone_hour_means = training_pickups.groupby(training_pickups.index.hour).mean()
previous_times = shock_pickups.index - pd.Timedelta(hours = 1)
previous_demand = shock_pickups.reindex(previous_times).to_numpy()
previous_baseline = past_hour_means.reindex(previous_times).to_numpy(copy = True)
is_evaluation = shock_pickups.index >= shock_training_end
previous_baseline[is_evaluation] = zone_hour_means.reindex(
    previous_times[is_evaluation].hour
).to_numpy()

# A positive shock means demand above the zone's usual level at that hour.
previous_shocks = previous_demand - previous_baseline
shock_neighbor_counts = adjacency.sum(axis = 1)
shock_has_neighbors = shock_neighbor_counts > 0

# Missing history should affect only zones connected to the missing neighbour.
neighbor_shocks = np.nan_to_num(previous_shocks, nan = 0.0) @ adjacency.T
missing_neighbor_history = (
    np.isnan(previous_shocks).astype(float) @ adjacency.T
) > 0
neighbor_shocks[:, shock_has_neighbors] /= shock_neighbor_counts[shock_has_neighbors]
neighbor_shocks[missing_neighbor_history] = np.nan
neighbor_shocks[:, ~shock_has_neighbors] = 0

shock_row_positions = shock_pickups.index.get_indexer(X["tpep_pickup_datetime"])
shock_col_positions = shock_pickups.columns.get_indexer(X["PULocationID"])
assert (shock_row_positions >= 0).all() and (shock_col_positions >= 0).all()
X["neighbors_mean_shock_1h"] = neighbor_shocks[shock_row_positions, shock_col_positions]
print("Added neighbors_mean_shock_1h: positive = above usual demand; negative = below.")
print("Missing history stays NaN; zones without neighbours use 0.")
X[["tpep_pickup_datetime", "PULocationID", "neighbors_mean_shock_1h"]].head()


In [ ]:
X


In [ ]:
X_shock = X.drop(columns = ["neighbors_mean_PU"])

X_train, X_val, X_test, y_train, y_val, y_test, train, val, test = data.split(X_shock)


In [ ]:
model_shock = data.model(train)
model_basic = data.model(train.drop(columns = ["neighbors_mean_shock_1h"]))
train_ablation = train.drop(columns = ["month"])


In [ ]:
model_shock.fit(X_train, y_train)


In [ ]:
y_pred_shock = model_shock.predict(val)

data.metrics(y_val, y_pred_shock)
